In [ ]:
# !pip install -q google-generativeai

In [ ]:
import pandas as pd
import numpy as np
import torch
import re

from sentence_transformers import SentenceTransformer, CrossEncoder, util
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# 1. Securely fetch the API key from Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
except userdata.SecretNotFoundError:
    print("Error: Please set the GEMINI_API_KEY in the Colab Secrets tab.")

# 2. Configure the Gemini API
genai.configure(api_key=api_key)

# 3. Initialize the chat model
llm_model = genai.GenerativeModel('gemini-2.5-flash')

print("Gemini API configured successfully!")

Gemini API configured successfully!


In [ ]:
# import google.generativeai as genai
# from google.colab import userdata

# # Authenticate
# genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# # List all available models for your key
# print("Available Models:")
# for m in genai.list_models():
#     if 'generateContent' in m.supported_generation_methods:
#         print(m.name)

In [ ]:
df = pd.read_csv("Coursera.csv")
df.head()

,Course Name,University,Difficulty Level,Course Rating,Course URL,Course Description,Skills
0,Write A Feature Length Screenplay For Film Or ...,Michigan State University,Beginner,4.8,https://www.coursera.org/learn/write-a-feature...,Write a Full Length Feature Film Script In th...,Drama Comedy peering screenwriting film D...
1,Business Strategy: Business Model Canvas Analy...,Coursera Project Network,Beginner,4.8,https://www.coursera.org/learn/canvas-analysis...,"By the end of this guided project, you will be...",Finance business plan persona (user experien...
2,Silicon Thin Film Solar Cells,�cole Polytechnique,Advanced,4.1,https://www.coursera.org/learn/silicon-thin-fi...,This course consists of a general presentation...,chemistry physics Solar Energy film lambda...
3,Finance for Managers,IESE Business School,Intermediate,4.8,https://www.coursera.org/learn/operational-fin...,"When it comes to numbers, there is always more...",accounts receivable dupont analysis analysis...
4,Retrieve Data using Single-Table SQL Queries,Coursera Project Network,Beginner,4.6,https://www.coursera.org/learn/single-table-sq...,In this course you�ll learn how to effectively...,Data Analysis select (sql) database manageme...


In [ ]:
df = df[["Course Name", "Course Description", "Skills", "Course Rating", "Difficulty Level", "Course URL"]]

In [ ]:
df.columns = df.columns.str.replace(' ', '_').str.lower()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3522 entries, 0 to 3521
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   course_name         3522 non-null   object
 1   course_description  3522 non-null   object
 2   skills              3522 non-null   object
 3   course_rating       3522 non-null   object
 4   difficulty_level    3522 non-null   object
 5   course_url          3522 non-null   object
dtypes: object(6)
memory usage: 165.2+ KB


In [ ]:
df["course_rating"].value_counts()

,count
course_rating,
4.7,740
4.6,623
4.8,598
4.5,389
4.4,242
4.9,180
4.3,165
4.2,121
5,90


In [ ]:
df.duplicated().sum()

np.int64(98)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df["course_rating"] = pd.to_numeric(df["course_rating"], errors="coerce")
df.dropna(inplace=True)

In [ ]:
df["text"] = df["course_name"] + " " + df["course_description"] + " " + df["skills"]

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
df["text"] = df["text"].apply(preprocess_text)

In [ ]:
df.head()

,course_name,course_description,skills,course_rating,difficulty_level,course_url,text
0,Write A Feature Length Screenplay For Film Or ...,Write a Full Length Feature Film Script In th...,Drama Comedy peering screenwriting film D...,4.8,Beginner,https://www.coursera.org/learn/write-a-feature...,write a feature length screenplay for film or ...
1,Business Strategy: Business Model Canvas Analy...,"By the end of this guided project, you will be...",Finance business plan persona (user experien...,4.8,Beginner,https://www.coursera.org/learn/canvas-analysis...,business strategy business model canvas analys...
2,Silicon Thin Film Solar Cells,This course consists of a general presentation...,chemistry physics Solar Energy film lambda...,4.1,Advanced,https://www.coursera.org/learn/silicon-thin-fi...,silicon thin film solar cells this course cons...
3,Finance for Managers,"When it comes to numbers, there is always more...",accounts receivable dupont analysis analysis...,4.8,Intermediate,https://www.coursera.org/learn/operational-fin...,finance for managers when it comes to numbers ...
4,Retrieve Data using Single-Table SQL Queries,In this course you�ll learn how to effectively...,Data Analysis select (sql) database manageme...,4.6,Beginner,https://www.coursera.org/learn/single-table-sq...,retrieve data using singletable sql queries in...


In [ ]:
bi_encoder = SentenceTransformer("all-mpnet-base-v2")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
course_embeddings = bi_encoder.encode(
    df['text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
def recommend_courses(query: str, top_n: int = 5):
    # --- STEP 1: RETRIEVAL (Fast) ---
    # We retrieve MORE than we need (e.g., 50 candidates) to give the Cross-Encoder options to choose from.
    initial_k = 50

    query_embedding = bi_encoder.encode(query, convert_to_tensor=True)

    # We use semantic_search provided by sentence_transformers for efficiency
    hits = util.semantic_search(query_embedding, course_embeddings, top_k=initial_k)
    hits = hits[0] # Get the list of results

    # --- STEP 2: RE-RANKING (Accurate) ---
    # Prepare pairs for the Cross-Encoder: [[Query, Text1], [Query, Text2], ...]
    cross_inp = []
    for hit in hits:
        # Get the original text from the dataframe using the corpus_id (index)
        text = df.iloc[hit['corpus_id']]['text']
        cross_inp.append([query, text])

    # Predict scores (higher is better)
    cross_scores = cross_encoder.predict(cross_inp)

    # Attach the new scores to the hits
    for idx, score in enumerate(cross_scores):
        hits[idx]['cross_score'] = score

    # Sort by the Cross-Encoder score (Descending)
    hits = sorted(hits, key=lambda x: x['cross_score'], reverse=True)

    # --- STEP 3: FORMAT OUTPUT ---
    results = []
    # Now we only take the top_n requested by the user (e.g., top 5)
    for hit in hits[:top_n]:
        # Retrieve the row from the original DataFrame
        row = df.iloc[hit['corpus_id']]

        results.append({
            "course_name": row["course_name"],
            "difficulty": row["difficulty_level"],
            "rating": float(row["course_rating"]),
            "url": row["course_url"],
            "similarity_score": round(float(hit['score']), 3), # Original cosine score
            "cross_encoder_score": round(float(hit['cross_score']), 3) # New accurate score
        })

    return results

In [ ]:
# 1. Initialize the Chat Session OUTSIDE the function.
# Run this line only once when you want to start a fresh conversation.
chat_session = llm_model.start_chat(history=[])

def generate_conversational_recommendation(user_query: str):
    # 1. Retrieve the top courses using your existing function
    top_courses = recommend_courses(query=user_query, top_n=4)

    # 2. Format the courses into a text block
    context_text = ""
    for i, course in enumerate(top_courses):
        # Adjust these dictionary keys to exactly match your recommend_courses output
        context_text += f"Course {i+1}: {course['course_name']}\n"
        context_text += f"- Difficulty: {course.get('difficulty', 'N/A')}\n"
        context_text += f"- Rating: {course.get('rating', 'N/A')} stars\n"
        context_text += f"- Link: {course.get('url', 'N/A')}\n\n"

    # 3. Create the prompt instruction
    # We tweak the prompt slightly so the AI knows it's an ongoing chat.
    prompt = f"""
    The student just said: "{user_query}"

    Here are the top courses retrieved from our database that match their current request:
    {context_text}

    Your task:
    1. If this is the start of the conversation, greet the student. If it is a follow-up, continue the conversation naturally.
    2. Recommend the best options from the list above based on what they just asked.
    3. Briefly explain *why* these courses fit their needs.
    4. Provide the links so they can easily enroll.
    5. CRITICAL: Do NOT invent or recommend any courses that are not in the database list provided above.
    """

    # 4. Use chat_session.send_message instead of llm_model.generate_content
    # This automatically saves the user prompt and the AI's response to the history.
    response = chat_session.send_message(prompt)

    return response.text

In [ ]:
user_input = "give me 3 courses about Machine Learning"

# Instead of printing a raw dictionary, print the chat response!
chat_reply = generate_conversational_recommendation(user_input)

print(chat_reply)

Hello! I can certainly help you with that. Here are three excellent courses about Machine Learning from our database:

1.  **Machine Learning with Python**
    *   This beginner-friendly course is perfect for getting hands-on experience, focusing on practical applications of machine learning using Python, which is a widely used language in the field.
    *   Rating: 4.6 stars
    *   Link: https://www.coursera.org/learn/machine-learning-with-python

2.  **Data for Machine Learning**
    *   Understanding and preparing data is fundamental to machine learning success. This beginner-level course will equip you with essential skills for handling data relevant to ML projects.
    *   Rating: 4.4 stars
    *   Link: https://www.coursera.org/learn/data-machine-learning

3.  **Fundamentals of Machine Learning in Finance**
    *   If you have an interest in applying machine learning to a specific domain, this advanced course explores its foundational concepts within the context of finance. It's

In [ ]:
user_input = "can you give me 2 more"

# Instead of printing a raw dictionary, print the chat response!
chat_reply = generate_conversational_recommendation(user_input)

print(chat_reply)

It looks like the courses retrieved for "2 more" aren't related to Machine Learning. The database search for this follow-up request returned courses on topics like happiness, grammar, and writing.

Since you're looking for more courses on Machine Learning, I cannot recommend any relevant options from the current list. Would you like me to try searching for something else, or were you perhaps interested in courses on other topics?


In [ ]:
user_input = "what about sql courses give me like 3 courses at sql"

# Instead of printing a raw dictionary, print the chat response!
chat_reply = generate_conversational_recommendation(user_input)

print(chat_reply)

Certainly! Switching gears to SQL, I can recommend three great courses for you from our database:

1.  **Databases and SQL for Data Science**
    *   This course is excellent for gaining a solid understanding of databases and SQL, specifically tailored for data science applications. It's listed as 'Conversant' difficulty, making it a strong option for building foundational skills.
    *   Rating: 4.6 stars
    *   Link: https://www.coursera.org/learn/sql-data-science

2.  **Advanced Relational Database and SQL**
    *   If you're looking to dive deeper into SQL and relational database concepts, this advanced course offers comprehensive content. It's perfect for those who want to master more complex queries and database management.
    *   Rating: 4.6 stars
    *   Link: https://www.coursera.org/learn/Advanced-rdb-sql

3.  **Intermediate Relational Database and SQL**
    *   This course serves as a great step to bridge the gap between basic SQL and the advanced topics. It focuses on int

In [ ]:
import os
import torch
import shutil
from google.colab import files

# 1. Create a temporary folder inside Colab
save_path = "flask_assets"
os.makedirs(save_path, exist_ok=True)

print("1. Saving models and data to Colab...")
# Save everything into the temporary folder
bi_encoder.save(f"{save_path}/bi_encoder_model")
cross_encoder.save(f"{save_path}/cross_encoder_model")
torch.save(course_embeddings, f"{save_path}/course_embeddings.pt")
df.to_pickle(f"{save_path}/cleaned_coursera_data.pkl")

print("2. Zipping the files together...")
# Compress the folder into a single .zip file named "flask_assets_bundle.zip"
shutil.make_archive("flask_assets_bundle", 'zip', save_path)

print("3. Triggering browser download! Please wait a moment...")
# This tells your web browser to download the file to your computer
files.download("flask_assets_bundle.zip")

1. Saving models and data to Colab...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2. Zipping the files together...
3. Triggering browser download! Please wait a moment...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>